# Исследование надежности заемщиков


## Открытие таблицы и изучение общей информации о данных

**Шаг 1.** Загрузка данных

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Шаг 2.** Первичный просмотр данных

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Шаг 3.** Изучение структуры датафрейма

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Шаг 4.** Анализ пропущенных значений

In [3]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Шаг 5.** Первичная обработка пропусков

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Шаг 6.** Обнаружение аномальных значений

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Шаг 7.** Анализ трудового стажа по типам занятости

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

**Шаг 8.** Изучение уникальных значений столбца `children`

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Шаг 9.** Удаление аномальных значений в столбце `children`

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Шаг 10.** Проверка результатов очистки

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Шаг 11.** Заполнение пропусков медианными значениями

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Шаг 12.** Контроль качества заполнения пропусков

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Шаг 13.** Преобразование типов данных

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Шаг 14.** Обработка неявных дубликатов

In [14]:
data['education'] = data['education'].str.lower()

**Шаг 15.** Удаление строк-дубликатов

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Шаг 16.** Создание категориального признака уровня дохода

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Шаг 17.** Анализ целей кредитования

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Шаг 18.** Создание категориального признака цели кредита

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [22]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.

# узнаем сколько всего семей в разрезе кол-ва детей
paid_all = data['children'].value_counts()
print(f'Всего семей/людей с детьми и без: \n{paid_all}')
print()

# Люди которые справились в срок
paid_counts = data[data['debt']==0]['children'].value_counts()


# Люди которые не справились в срок
not_paid_counts = data[data['debt']==1]['children'].value_counts()

# Соберем два списка вместе и посмотрим разницу
df_counts = pd.DataFrame({"Погасили кредит": paid_counts, "Не погасили": not_paid_counts, "Погасили кредит %": paid_counts/paid_all*100, "Не погасили %": not_paid_counts/paid_all*100}).round(2).fillna(0)
print(df_counts)

Всего семей/людей с детьми и без: 
0    14091
1     4808
2     2052
3      330
4       41
5        9
Name: children, dtype: int64

   Погасили кредит  Не погасили  Погасили кредит %  Не погасили %
0            13028       1063.0              92.46           7.54
1             4364        444.0              90.77           9.23
2             1858        194.0              90.55           9.45
3              303         27.0              91.82           8.18
4               37          4.0              90.24           9.76
5                9          0.0             100.00           0.00


**Вывод:** Как видно из таблицы, количество погасившиих кредит больше чем не погасивших. 
Это говорит нам что количество детей в семье не влияет на то сможет ли люди погасить кредит вовремя или нет. Даже семьи у которых 5 детей смогли погасить кредит вовремя, это говорит что даже большое количество детей не помешало вовремя вернуть кредит.

In [23]:
def p_table_create(category):
  table = data.groupby(category)['debt'].agg(['count', 'sum', 'mean'])
  table.columns = ['total_clients', 'debtors', 'debt_rate']
  table['share_of_sample'] = table['total_clients'] / data.shape[0]
  table = table.sort_values(by='debt_rate', ascending=False)
  table['debt_rate'] = table['debt_rate'].apply(lambda x: "{0:.2%}".format(x))
  table['share_of_sample'] = table['share_of_sample'].apply(lambda x: "{0:.2%}".format(x))
  return table

In [24]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
# Группируем категории у которых меньше 50 колва людей
def age_group(a):
    if a >= 3:
        return '3-5'
    else:
        return a

data['children_grouped'] = data['children'].apply(age_group)


children_pivot = p_table_create(data['children_grouped'])
children_pivot


,count,sum,mean,proportion_of_group
children_grouped,,,,
0,14091,1063,7.54%,66.06%
1,4808,444,9.23%,22.54%
2,2052,194,9.45%,9.62%
3-5,380,31,8.16%,1.78%


**Вывод:** По логике долговозвращающей группой должна быть та в которой от 3 до 5 и более детей, однако это группы с детьми от 1 до 2. Даже не знаю с чем это связано.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [25]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.
# Общее количество в группах
fam_counts = data['family_status'].value_counts()

# получаем количество оплативших кредит в срок 
fam_paid = data.loc[data['debt'] == 0]['family_status'].value_counts()

# получаем количество не оплативших кредит в срок
fam_not_paid = data.loc[data['debt'] == 1]['family_status'].value_counts()

# собираем три переменные в единый DataFrame
fam_df = pd.DataFrame({'Всего в группе людей':fam_counts, 'Погасили кредит':fam_paid, 'Не погасили':fam_not_paid, "Погасили кредит %":fam_paid/fam_counts*100, "Не погасили кредит %":fam_not_paid/fam_counts*100}).round(2).fillna(0)
display(fam_df)
print()

print('Процент всех людей вернувших кредит в срок, назависимо от семейного статуса:', fam_df['Погасили кредит %'].sum())
print('Процент всех людей не вернувших кредит в срок, назависимо от семейного статуса:', fam_df['Не погасили кредит %'].sum())


,Всего в группе людей,Погасили кредит,Не погасили,Погасили кредит %,Не погасили кредит %
женат / замужем,12261,11334,927,92.44,7.56
гражданский брак,4134,3749,385,90.69,9.31
Не женат / не замужем,2796,2523,273,90.24,9.76
в разводе,1189,1105,84,92.94,7.06
вдовец / вдова,951,888,63,93.38,6.62



Процент всех людей вернувших кредит в срок, назависимо от семейного статуса: 459.69
Процент всех людей не вернувших кредит в срок, назависимо от семейного статуса: 40.31


**Вывод:** Как мы можем видить, семейный статус влияет в незначительной степени. Это можно наблюдать на основе каждой группы. <br> Например для группы "Женат / замужем" процент не оплативших составляет всего лишь 8.18%.

In [26]:
# Создаем сводную таблицу
family_status_data = p_table_create(data['family_status'])
family_status_data

,count,sum,mean,proportion_of_group
family_status,,,,
женат / замужем,12261,927,7.56%,57.48%
гражданский брак,4134,385,9.31%,19.38%
Не женат / не замужем,2796,273,9.76%,13.11%
в разводе,1189,84,7.06%,5.57%
вдовец / вдова,951,63,6.62%,4.46%


**Вывод:** Проблемы видны у тех:
* кто не женат/не замужем, скорей всего они в одиночку не справляются с кредитом, были бы они в браке то таких проблем возможно не было бы :)
* кто состоит в гражданском браке

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [27]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.

# Общее количество в группах
income_counts = data['income_type'].value_counts()

# получаем количество оплативших кредит в срок 
income_paid = data.loc[data['debt'] == 0]['income_type'].value_counts()

# получаем количество не оплативших кредит в срок
income_not_paid = data.loc[data['debt'] == 1]['income_type'].value_counts()

# собираем три переменные в единый DataFrame
income_df = pd.DataFrame({'Всего в группе людей':income_counts, 'Погасили кредит':income_paid, 'Не погасили':income_not_paid, "Погасили кредит %":income_paid/income_counts*100, "Не погасили кредит %":income_not_paid/income_counts*100}).round(2).fillna(0)

# выводим результат
display(income_df.sort_values(by='Не погасили кредит %', ascending=False))
print()

,Всего в группе людей,Погасили кредит,Не погасили,Погасили кредит %,Не погасили кредит %
в декрете,1,0.0,1.0,0.00,100.00
безработный,2,1.0,1.0,50.00,50.00
сотрудник,11015,9961.0,1054.0,90.43,9.57
компаньон,5047,4673.0,374.0,92.59,7.41
госслужащий,1451,1365.0,86.0,94.07,5.93
пенсионер,3812,3596.0,216.0,94.33,5.67
предприниматель,2,2.0,0.0,100.00,0.00
студент,1,1.0,0.0,100.00,0.00


**Вывод:** Уровень дохода в срок повлиял только на слеующие группы: в декрете, безработный. На остальные группы уровень дохода повлиял лишь в небольшой степени.

In [28]:
# Создаем сводную таблицу
income_data = p_table_create(data['total_income_category'])

# Группы с меньшим количеством отпадают
income_data = income_data[income_data['total_clients'] > 100].dropna()

income_data

,count,sum,mean,proportion_of_group
total_income_category,,,,
C,15921,1353,8.50%,74.64%
B,5014,354,7.06%,23.51%
D,349,21,6.02%,1.64%


**Новый вывод:** Люди с уровнем дохода С состовляют большую часть выборки и их возврат кредита в срок равен 8.5%. Значит уровень дохода также влияет на возврат кридета в срок

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [29]:
# Ваш код будет здесь. Вы можете создавать новые ячейки.

# Общее количество в группах
purpose_counts = data['purpose_category'].value_counts()

# получаем количество оплативших кредит в срок 
purpose_paid = data.loc[data['debt'] == 0]['purpose_category'].value_counts()

# получаем количество не оплативших кредит в срок
purpose_not_paid = data.loc[data['debt'] == 1]['purpose_category'].value_counts()

# собираем три переменные в единый DataFrame
purpose_df = pd.DataFrame({'Всего в группе людей':purpose_counts, 'Погасили кредит':purpose_paid, 'Не погасили':purpose_not_paid, "Погасили кредит %":purpose_paid/purpose_counts*100, "Не погасили кредит %":purpose_not_paid/purpose_counts*100}).round(2).fillna(0)

# выводим результат
display(purpose_df.sort_values(by='Не погасили кредит %', ascending=False))
print()

,Всего в группе людей,Погасили кредит,Не погасили,Погасили кредит %,Не погасили кредит %
операции с автомобилем,4279,3879,400,90.65,9.35
получение образования,3988,3619,369,90.75,9.25
проведение свадьбы,2313,2130,183,92.09,7.91
операции с недвижимостью,10751,9971,780,92.74,7.26


**Вывод:** Вывод как и прежде, цели получения кредита не повлияли в значительной степени на то будут ли они возвращены в срок или нет.

In [30]:
# Создаем сводную таблицу
purpose_data = p_table_create(data['purpose_category'])

purpose_data


,count,sum,mean,proportion_of_group
purpose_category,,,,
операции с недвижимостью,10751,780,7.26%,50.40%
операции с автомобилем,4279,400,9.35%,20.06%
получение образования,3988,369,9.25%,18.70%
проведение свадьбы,2313,183,7.91%,10.84%


**Вывод:** Две категории: Операции с автомобилем и получение образования имеют высокие показатели среди остальных. Значит, заемщики чьи цели связаны с этими двумя склонны задерживать возврат. Итог: Цели имеют связь с возвратом кредита в срок.

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

**Ответ:** Проблемы при выгрузке данных. А также, возможно данные не были внесены при вводе. Также, при сохранение данных. Возможны случаи изменения типов данных, при этом числовые данные преобразованные в текстовые, могут быть искажены.

**Дополнение:** В работе с данными почти всегда вас ждут сюрпризы:

- Почему-то выгрузили не те данные или не всё, что есть.
- Ошибки в алгоритмах: скажем, время учли неверно.
- Не тот формат: например, вместо секунд записали минуты.
- Упущен какой-нибудь существенный факт.

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* Проблема использования среднего значения: Если у вас 10 яблок а у Пети 0, значит у них обоих в среднем по 5 яблок. Так что это не всегда удачное решение использовать ср. зн

**Новый ответ**: Медианой лучше всего заполнять потому что он не реагирует на аномалии(большие и маленькие выбросы)

### Шаг 4: общий вывод.

### Напишите ваш общий вывод. ВЫВОД v2 ### 
**Вывод после поиска зависимостей между несколькими факторами и возвратом кредита в срок. В качестве факторов были использованы такие факторы как:**
1.	**Количество детей в семье.** Данные о количестве детей находятся в диапазоне от 0 до 5. Кол-во 4х детных семей составляет 41, а 5 детных 9, поэтому они были включены в группу «3-5» и общее кол-во составило 380;
2.	**Семейное положение.** Присутствуют 5 типов семейного положения: женат / замужем, гражданский брак, вдовец / вдова, в разводе, Не женат / не замужем;
3.	**Уровень дохода.** Есть 5 уровней дохода: A, B, C, D, E. Были убраны категории А и Е изза меньшего кол-во данных;
4.	**Цели получения кредита.**  Целей было много однако после группировки получили всего 4: операции с недвижимостью, операции с автомобилем, получение образования, проведение свадьбы.
<br>
**Анализ каждого из факторов.**
* **Количество детей в семье.**
При анализе зависимости между возвратом кредита с количеством детей была выявлена связь. 4 группы имеют следующие показатели (в порядке убывания показателя возврата кредита):
 1.	**2 детные.** Люди с 2мя детьми имеют показатель 9.45%, 194 из 2052 не возвращают кредит в срок. Данная группа состовляет 9.62% из всей выборки.

 2.	**1 детные.** 444 из 4808 что составляет 9.23%. Данная группа состовляет 22,54% из всей выборки.

 3.	**3-5 детные.** 31 из 380, 8.16%. Данная группа состовляет 1,78% из всей выборки.

 4.	**0.** Бездетные люди имеют показатель 7.54%, 1063 из 14091. Данная группа состовляет 66,06% из всей выборки.

<br>
<b>Вывод.</b> Как видно из анализа 2 и 1 детные люди имеют большие показатели 9.45% и 9.23%. Можно сказать, что количество влияет на то, как быстро люди вернут кредит в срок. 
<br>

* **Семейное положение.**
Данный фактор также имеет связь. 5 групп имеют следующие показатели (в порядке убывания показателя возврата кредита):
 1.	Не женат / не замужем. Эта группа составляет 2796 (13.11% из всей выборки). 
273 (9.76%) не возвращают кредит в срок.
 2.	Гражданский брак. Эта группа составляет 4134 (19.38% из всей выборки). 
385 (9.31%) не возвращают кредит в срок.
 3.	Женат / замужем. Эта группа составляет 12261 (57.48% из всей выборки). 
927 (7.56%) не возвращают кредит в срок.
 4.	В разводе. Эта группа составляет 1189 (5.57% из всей выборки). 
84 (7.06%) не возвращают кредит в срок.
 5.	Вдовец / вдова. Эта группа составляет 951 (4.46% из всей выборки). 
63 (6.62%) не возвращают кредит в срок.

<br>
<b>Вывод.</b> Данный фактор также имеет связь с возвратом кредита в срок. Группы «Не женат / не замужем» и «Гражданский брак» имеют самые высокие показатели 9.76 и 9.31 процентов. Также стоит обратить внимание на группу «Женат / замужем» эта группа составляет 57.48% из всей выборки и имеет 7.56%.  Итог, семейное положение влияет на возврат долга. Из вышеуказанных показателей можно сказать, что на столько и влияет семейное положение.
<br>

* **Уровень дохода.**
Уровень дохода также имеет связь с возвратом кредита в срок, 3 группы имеют следующие показатели (в порядке убывания показателя возврата кредита):
 1.	С. Эта группа составляет 15921 (74.64% из всей выборки). 
1353 (8.50%) не возвращают кредит в срок.
 2.	В. Эта группа составляет 5014 (23.51% из всей выборки). 
354 (7.06%) не возвращают кредит в срок.
 3.	D. Эта группа составляет 349 (1.64% из всей выборки). 
349 (6.02%) не возвращают кредит в срок.

<br>
<b>Вывод.</b> Данный фактор также имеет связь с возвратом кредита в срок. Из категории С 8.50% не возвращают кредит в срок. Также, уровень дохода повлиял на категорию, где у 7.06% проблемы с возвратом в срок. Категория D составляет менее 2% поэтому учитывать ее показатель не целесообразно.
<br>

* **Цели кредита.**
При анализе зависимости между возвратом кредита с количеством детей была выявлена связь. 4 группы имеют следующие показатели (в порядке убывания показателя возврата кредита):
 1.	операции с автомобилем. Всего 4279 (Из всей выборки составляет 20.06%) данных, 400 (9.35%) из которых не возвращают кредит в срок.
 2.	получение образования. Всего 3988 (Из всей выборки составляет 18.70%) данных, 369 (9.25%) из которых не возвращают кредит в срок.
 3.	проведение свадьбы. Всего 2313 (Из всей выборки составляет 10.84%) данных, 
183 (7.91%) из которых не возвращают кредит в срок.
 4.	операции с недвижимостью. Всего 10751 (Из всей выборки составляет 50.40%) данных, 780 (7.26%) из которых не возвращают кредит в срок.
 
<br>
<b>Вывод.</b> Как видно из анализа 2 и 1 детные люди имеют большие показатели 9.45% и 9.23%. Можно сказать, что количество влияет на то, как быстро люди вернут кредит в срок. 

 

## Ссылка на репозиторий

https://github.com/JasurbekAkhunov/credit-risk-analysis